[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ALighieriDxG/verifipay-ep1/blob/main/demo_ep1.ipynb)

# Evaluación Parcial 1 — VerifiPay
## Demo en Python, celda por celda

Caso: fintech ficticia chilena. El agente no contesta de memoria. Recupera un documento, una norma o una transacción, y recién entonces redacta. Si no hay evidencia, escala a un operador.

Ejecuta las celdas de arriba hacia abajo. Cada bloque imprime lo que después se muestra en la defensa.

| Pieza | Dónde vive | Qué hace |
|---|---|---|
| Corpus | `data/` | Manual, FAQ, normas y tickets simulados |
| Índices | FAISS, dos archivos | Separa operación y normativa |
| Agente | `src/agent.py` | Elige la herramienta |
| Prompt | `src/prompts.py` | Obliga a citar y prohíbe pedir la tarjeta |

## 0. Preparar el proyecto

En local usa el `.env` de esta carpeta. La key no se imprime.
En Colab hay que guardar `LLM_API_KEY` en Secretos (ícono de llave) y volver a ejecutar esta celda.

In [1]:
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    get_ipython().system("git clone --depth 1 https://github.com/ALighieriDxG/verifipay-ep1.git")
    os.chdir("verifipay-ep1")
    get_ipython().system("pip install -q -r requirements.txt")
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")

candidatos = [
    Path.cwd(),
    Path.cwd() / "GITHUB" / "BOT_SOPORTE",
    Path(r"e:/ing soluciones/GITHUB/BOT_SOPORTE"),
]
raiz = next(p for p in candidatos if (p / "src" / "agent.py").exists())
os.chdir(raiz)
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from dotenv import load_dotenv
load_dotenv(raiz / ".env")

tiene_key = bool(os.getenv("LLM_API_KEY"))
print("Proyecto:", raiz)
print("LLM_API_KEY cargada:", "sí" if tiene_key else "no — completa el archivo .env")

Proyecto: e:\ing soluciones\GITHUB\BOT_SOPORTE
LLM_API_KEY cargada: sí


## 1. Ver el corpus, sin llamar al modelo

Esto solo lee archivos. Sirve para mostrar que hay dos mundos de datos: el interno (cómo opera VerifiPay) y el externo (qué exige la norma).

In [2]:
from src.ingest import load_external_documents, load_internal_documents

internos = load_internal_documents()
externos = load_external_documents()

print(f"Documentos internos: {len(internos)}")
for nombre in sorted({d.metadata['source'] for d in internos}):
    print("  -", nombre)

print(f"\nDocumentos externos: {len(externos)}")
for nombre in sorted({d.metadata['source'] for d in externos}):
    print("  -", nombre)

Documentos internos: 13
  - faq_comercios.json
  - glosario_fintech.md
  - manual_soporte.md
  - politicas_comerciales.md
  - tickets_resueltos.csv

Documentos externos: 3
  - ley_19628_resumen.md
  - normativa_cmf.md
  - pci_dss_resumen.md


## 2. Cargar los índices FAISS

Si ya corriste `python scripts/build_index.py`, esta celda reutiliza esos vectores. Si no existen, los construye.

Dos índices a propósito: una pregunta de la terminal no debe recuperar un párrafo de PCI-DSS solo porque ambas hablan de “tarjeta”.

In [3]:
from src.ingest import ensure_indexes

indice_interno, indice_externo = ensure_indexes(force=False)
print("Vectores internos:", indice_interno.index.ntotal)
print("Vectores externos:", indice_externo.index.ntotal)

Vectores internos: 24
Vectores externos: 11


## 3. Qué fragmento recupera la búsqueda

Antes de pedirle una redacción al modelo, se mira el fragmento. En la defensa se compara esta evidencia con la frase final: tienen que coincidir.

In [4]:
from src.rag import retrieve_with_scores

def mostrar_recuperacion(indice, pregunta, titulo):
    resultado = retrieve_with_scores(indice, pregunta)
    print("=" * 72)
    print(titulo)
    print("Pregunta:", pregunta)
    print(f"Confianza del mejor fragmento: {resultado.max_score:.2f}  (umbral de escalamiento: 0.35)")
    print()
    print(resultado.context)
    print()

mostrar_recuperacion(
    indice_interno,
    "¿Cuándo me depositan las ventas del fin de semana?",
    "RAG INTERNO — abono de fin de semana",
)
mostrar_recuperacion(
    indice_externo,
    "¿Puedo pedir el número de tarjeta del cliente por chat?",
    "RAG EXTERNO — datos de tarjeta",
)

RAG INTERNO — abono de fin de semana
Pregunta: ¿Cuándo me depositan las ventas del fin de semana?
Confianza del mejor fragmento: 0.81  (umbral de escalamiento: 0.35)

[Fragmento 1 | Fuente: faq_comercios.json]
Pregunta: ¿Cuándo me depositan las ventas del fin de semana?
Respuesta: Las ventas de viernes, sábado y domingo se depositan el martes siguiente antes de las 14:00 hrs.

[Fragmento 2 | Fuente: manual_soporte.md]
## 3. Conciliación de ventas

Las ventas con tarjeta de débito se abonan en **24 horas hábiles**. Las ventas con tarjeta de crédito se abonan en **48 horas hábiles**. Ventas realizadas viernes, sábado o domingo se depositan el **martes siguiente** antes de las 14:00 hrs.

## 4. Devoluciones

[Fragmento 3 | Fuente: politicas_comerciales.md]
## Plazos de abono

- Débito: 24 horas hábiles
- Crédito: 48 horas hábiles
- Fin de semana: depósito martes antes de 14:00 hrs

## Devoluciones

Plazo máximo del comercio: 5 días hábiles desde la solicitud del cliente final. VerifiPay n

## 4. La API de transacciones no usa el índice

El estado de una venta no está en un PDF. La herramienta busca el ID en un diccionario simulado. Si el ID no existe, lo dice. No completa el registro.

In [5]:
from src.api_mock import consultar_transaccion

for tx in ["TX-20250908-001", "TX-20250908-002", "TX-NO-EXISTE"]:
    print(consultar_transaccion(tx))
    print()

Transacción TX-20250908-001: estado=abonado, monto=$125,000 CLP, medio=crédito, fecha_abono=2025-09-10, comercio=Café Central.

Transacción TX-20250908-002: estado=pendiente, monto=$48,900 CLP, medio=débito, fecha_abono=2025-09-09, comercio=Ferretería Norte.

No se encontró la transacción TX-NO-EXISTE. Verifique el ID en el panel de conciliación.



## 5. Armar el agente

LangChain le entrega tres herramientas al modelo:

- `buscar_documentacion_interna` — plazos, comisiones, errores, FAQ
- `buscar_normativa_externa` — Ley 19.628, CMF, PCI-DSS
- `consultar_estado_transaccion` — ID de la forma `TX-...`

El modelo elige. El prompt le exige citar la fuente y no inventar cifras. Temperatura `0.1`.

La demo usa `ministral-8b-latest`. Es el mismo proveedor (Mistral) y acepta herramientas. `mistral-small-latest` está en el diseño, pero el plan gratuito responde 429 (límite de consultas) y por eso la celda de preguntas se caía.

In [10]:
import importlib
import src.agent as agent_mod
importlib.reload(agent_mod)
from src.agent import ask, build_agent_executor

agente = build_agent_executor(force_reindex=False, model="ministral-8b-latest")

def demostrar(pregunta):
    print("=" * 72)
    print("CONSULTA:", pregunta)
    print("-" * 72)
    respuesta = ask(agente, pregunta)
    print(respuesta.get("output", "Sin respuesta"))
    print()

print("Agente listo. Herramientas:", [t.name for t in agente.tools])

TypeError: build_agent_executor() got an unexpected keyword argument 'model'

## 6. Las cuatro consultas de la defensa

Qué tienes que señalar en voz alta:

1. Fin de semana → martes antes de las 14:00, citado en el manual o el FAQ.
2. Error E-204 → falla de conexión, Wi-Fi y reinicio. Si pasa de 30 minutos, nivel 2.
3. `TX-20250908-002` → pendiente, $48.900, débito, Ferretería Norte. La fuente es la API, no un documento.
4. Número de tarjeta → no se pide. PAN y CVV quedan fuera del chat.

In [7]:
demostrar("¿Cuándo me depositan las ventas del fin de semana?")

CONSULTA: ¿Cuándo me depositan las ventas del fin de semana?
------------------------------------------------------------------------


OpenAIRateLimitError: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}

In [8]:
demostrar("Mi terminal dice error E-204, ¿qué hago?")

CONSULTA: Mi terminal dice error E-204, ¿qué hago?
------------------------------------------------------------------------


OpenAIRateLimitError: Error code: 429 - {'object': 'error', 'message': 'Rate limit exceeded', 'type': 'rate_limited', 'param': None, 'code': '1300', 'raw_status_code': 429}

In [ ]:
demostrar("Consulta el estado de TX-20250908-002")

In [ ]:
demostrar("¿Puedo pedir el número de tarjeta del cliente por chat?")

## Cómo leerlo en la presentación

La celda 3 muestra el fragmento recuperado. Las celdas 6 a 9 muestran la redacción del modelo. Si el plazo o el monto no están en el fragmento, la respuesta está mal aunque suene creíble.

El prototipo demuestra cita, elección de herramienta y escalamiento. No calcula todavía el 85 % sobre 350 consultas reales: la empresa y los datos son ficticios.